[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb03_obs_action_reward.ipynb)

# NB03 — Obs / Action / Reward

> **Prerequisites**: [NB01 — Quickstart](./nb01_quickstart.ipynb)  
> **Time**: ~15 minutes

## What You'll Learn

1. What each dimension of the observation vector represents (and why)
2. How the action space is structured and how clipping works
3. How the reward is decomposed into cost, penalty, and bonus terms
4. Wrapper flow diagram: which wrapper to reach for in each scenario

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper, MARLWrapper
from powerzoo.wrappers.flatten import FlattenWrapper

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

## 1. Observation Space

The observation is a **flat numpy vector** (after wrapping). Its dimensions come from
global grid state and local agent state, concatenated together.

We'll use Task 1 (OPF) as the primary example.

In [ ]:
env = make_task_env("marl_opf", split="train")
gym_env = GymnasiumWrapper(env)

obs, info = gym_env.reset(seed=42)

print(f"Observation shape: {obs.shape}")
print(f"Observation dtype: {obs.dtype}")
print(f"\nFirst 10 values: {obs[:10]}")

# Obs names are available on the wrapped env
if hasattr(gym_env, 'obs_names'):
    print(f"\nObs dimension names:")
    for i, name in enumerate(gym_env.obs_names[:10]):
        print(f"  [{i:2d}] {name:30s} = {obs[i]:.4f}")
    if len(gym_env.obs_names) > 10:
        print(f"  ... ({len(gym_env.obs_names)} total dimensions)")

### Observation structure by task

| Group | Dimensions | Physical meaning | ML analogy |
|---|---|---|---|
| **Global: grid state** | varies | Node load injections (normalized) | Environment physics |
| **Global: line flows** | n_lines | Power flow on each transmission line (p.u.) | Constraint state |
| **Global: time features** | 2 | `sin(2π·hour/24)`, `cos(2π·hour/24)` | Periodic time embedding |
| **Local: agent identity** | 1 | Unit index (one-hot or integer) | Agent ID |
| **Local: power limits** | 2 | `p_min`, `p_max` for this generator | Action bounds |
| **Local: cost coeffs** | 2 | Marginal cost coefficients | Objective function |
| **Local: SOC** (Task 2/3) | 1 | State of charge ∈ [0, 1] | Continuous state |

> **Power systems analogy for ML readers**: the grid state is like the joint configuration
> in a robotics task — it captures the current system state that the agents must react to.
> Line flows are like joint torques — they represent constraint loads that must stay within bounds.

In [ ]:
# Compare obs shapes across all three tasks
print("Observation shapes per task:")
print(f"{'Task':30s} {'Gym obs dim':12s} {'Agents':8s} {'Act dim/agent':14s}")
print("-" * 65)

for task_name in ["marl_opf", "marl_der_arbitrage", "marl_ev_v2g"]:
    e = make_task_env(task_name, split="train")
    g = GymnasiumWrapper(e)
    obs, _ = g.reset(seed=0)
    act_dim = g.action_space.shape[0]
    n_agents = len(e.agents)
    print(f"{task_name:30s} {obs.shape[0]:12d} {n_agents:8d} {act_dim:14d}")

## 2. Action Space

All PowerZoo tasks have **continuous Box action spaces**.

- **Task 1 (OPF)**: each unit outputs a power allocation score ∈ [0, 1].  
  The scores are softmax-normalized and scaled to physical MW output.
- **Task 2 (DER)**: each battery outputs charge/discharge power ∈ [−P_max, +P_max] MW.  
  Positive = charge, Negative = discharge (V2G).
- **Task 3 (EV)**: same as Task 2, but with an availability mask (action is zeroed when EV is away).

In [ ]:
env_opf = make_task_env("marl_opf", split="train")
gym_opf = GymnasiumWrapper(env_opf)

act_space = gym_opf.action_space
print(f"Action space: {act_space}")
print(f"  Low:  {act_space.low}")
print(f"  High: {act_space.high}")
print(f"  Dtype: {act_space.dtype}")

# Out-of-bounds actions are clipped automatically
obs, info = gym_opf.reset(seed=0)
extreme_action = np.array([2.0] * act_space.shape[0])  # deliberately out of bounds
obs_next, reward, term, trunc, info = gym_opf.step(extreme_action)
print(f"\nClipping demo: action={extreme_action[0]:.1f} → reward={reward:.3f} (no crash)")

In [ ]:
# Action distribution: random vs structured
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# Random actions
random_acts = np.array([act_space.sample() for _ in range(500)])
axes[0].hist(random_acts.flatten(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Random actions (uniform sampling)')
axes[0].set_xlabel('Action value')
axes[0].set_ylabel('Count')

# Softmax-style actions (more realistic)
softmax_acts = np.exp(np.random.randn(500, act_space.shape[0]))
softmax_acts /= softmax_acts.sum(axis=1, keepdims=True)
axes[1].hist(softmax_acts.flatten(), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Softmax-distributed actions (realistic)')
axes[1].set_xlabel('Action value (= power allocation score)')

plt.suptitle('Action distributions — Task 1 (OPF)', y=1.02)
plt.tight_layout()
plt.show()

## 3. Reward Decomposition

The reward is composed of multiple terms. Understanding each term tells you *why* the
agent behaves as it does — and which constraints it might be violating.

### Task 1 (OPF) reward

$$r_t = -\underbrace{\sum_i c_i \cdot p_i}_{\text{generation cost}} - \lambda_{\text{safety}} \cdot \underbrace{\sum_l \max(0, |f_l| - f_l^{\max})}_{\text{line overload penalty}}$$

- `cost_weight = 1.0`   (from task config)
- `safety_weight = 0.5`

### Task 2 (DER) reward

$$r_t = \underbrace{\text{price}(t) \cdot \text{discharge}(t)}_{\text{arbitrage profit}} - \lambda_{\text{soc}} \cdot \underbrace{(\text{SOC} - \text{SOC}_{\text{target}})^2}_{\text{SOC deviation}} - \lambda_{\text{volt}} \cdot \underbrace{\text{voltage violation}}_{\text{grid safety}}$$

### Task 3 (EV) reward

Same as Task 2 plus a **departure penalty**:

$$r_t \mathrel{-}= \lambda_{\text{depart}} \cdot \max(0, \text{SOC}_{\text{min\_depart}} - \text{SOC at departure})$$

In [ ]:
# Run one episode and inspect info dict for reward components
env_der = make_task_env("marl_der_arbitrage", split="train")
gym_der = GymnasiumWrapper(env_der)

obs, info = gym_der.reset(seed=42)
all_infos = []

while True:
    action = gym_der.action_space.sample()
    obs, reward, terminated, truncated, info = gym_der.step(action)
    all_infos.append({'reward': reward, **info})
    if terminated or truncated:
        break

print(f"Episode length: {len(all_infos)} steps")
print(f"\ninfo dict keys at last step: {list(info.keys())}")
if info:
    for k, v in info.items():
        if not isinstance(v, dict):
            print(f"  {k}: {v}")

In [ ]:
# Plot reward over episode
rewards = [d['reward'] for d in all_infos]

fig, ax = plt.subplots(figsize=(9, 3))
ax.fill_between(range(len(rewards)), rewards, alpha=0.3, color='steelblue')
ax.plot(rewards, color='steelblue', linewidth=1.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Timestep')
ax.set_ylabel('Reward')
ax.set_title('Task 2 (DER Arbitrage) — step reward, random policy')
plt.tight_layout()
plt.show()

## 4. Wrapper Flow Diagram

Here is a decision guide for which wrapper to use:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.axis('off')

# Helper
def box(ax, x, y, w, h, text, fc='#e8f4fd', ec='#2196F3', fs=9):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                         facecolor=fc, edgecolor=ec, linewidth=1.5))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fs,
            fontweight='bold', color='#1a1a1a', wrap=True)

def arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my + 0.15, label, ha='center', va='bottom', fontsize=7.5, color='#555')

# Source
box(ax, 0.2, 2.0, 2.5, 1.0, 'make_task_env()\n(native MARL)', fc='#f3e5f5', ec='#9C27B0')

# Gym
box(ax, 3.5, 2.8, 2.5, 1.0, 'GymnasiumWrapper\n(gym.Env)', fc='#e8f5e9', ec='#4CAF50')
arrow(ax, 2.7, 2.7, 3.5, 3.2)

# Flatten
box(ax, 7.2, 2.8, 2.5, 1.0, 'FlattenWrapper\n(1D obs + act)', fc='#fff9c4', ec='#FFC107')
arrow(ax, 6.0, 3.2, 7.2, 3.2)

# SB3 etc.
box(ax, 10.2, 2.8, 1.5, 1.0, 'SB3 / CleanRL\nTianshou', fc='#fce4ec', ec='#E91E63', fs=8)
arrow(ax, 9.7, 3.2, 10.2, 3.2)

# MARL
box(ax, 3.5, 0.5, 2.5, 1.0, 'MARLWrapper\n(PettingZoo)', fc='#e8f4fd', ec='#2196F3')
arrow(ax, 2.7, 2.3, 3.5, 1.1)

# RLlib
box(ax, 7.2, 0.5, 2.5, 1.0, 'ParallelPettingZooEnv', fc='#fce4ec', ec='#E91E63', fs=8)
arrow(ax, 6.0, 1.0, 7.2, 1.0)

box(ax, 10.2, 0.5, 1.5, 1.0, 'RLlib\nMAPPO/IPPO', fc='#fce4ec', ec='#E91E63', fs=8)
arrow(ax, 9.7, 1.0, 10.2, 1.0)

# evaluate
box(ax, 3.5, 4.2, 2.5, 0.7, 'evaluate(policy, gym_env)', fc='#e8f4fd', ec='#607D8B', fs=8)
ax.annotate('', xy=(4.75, 4.2), xytext=(4.75, 3.8),
            arrowprops=dict(arrowstyle='<-', color='#607D8B', lw=1.2))
ax.text(5.2, 4.0, 'for benchmarking', fontsize=7, color='#607D8B')

ax.set_title('Wrapper routing — which layer for which use case', fontsize=11, pad=10)
plt.tight_layout()
plt.show()

## 5. FlattenWrapper in Detail

`FlattenWrapper` handles both obs and action flattening when the underlying
environment uses dict spaces. It records the exact slice for each key so you
can reconstruct individual components if needed.

In [ ]:
env_der = make_task_env("marl_der_arbitrage", split="train")
gym_der = GymnasiumWrapper(env_der)
flat_der = FlattenWrapper(gym_der)

obs_gym, _ = gym_der.reset(seed=0)
obs_flat, _ = flat_der.reset(seed=0)

print("Before FlattenWrapper:")
print(f"  obs type: {type(obs_gym).__name__}")
print(f"  obs space: {gym_der.observation_space}")

print("\nAfter FlattenWrapper:")
print(f"  obs type: {type(obs_flat).__name__}")
print(f"  obs shape: {obs_flat.shape}")
print(f"  obs space: {flat_der.observation_space}")

---

## Summary

| Concept | Key points |
|---|---|
| **Observation** | Flat vector: global grid state + local agent state; time features use sin/cos encoding |
| **Action** | Continuous Box; Task 1: allocation score [0,1]; Task 2/3: charge power ∈ [−P_max, P_max] |
| **Reward** | = −cost − constraint penalties; decompose via `info` dict for debugging |
| **FlattenWrapper** | Concatenates dict obs keys into 1D vector; stores slices for recovery |
| **Obs names** | `gym_env.obs_names` gives human-readable labels for each dimension |

## Next

→ [NB04 — Task 1: Multi-Agent Economic Dispatch (Full Walkthrough)](./nb04_task_opf.ipynb)